### Week 8: Persistence: Saving Objects

**Today you will:** save lists, dictionaries and whole objects to a file with `pickle`, load them back in a later run, and use a saved file as a cache.

Start by picking up this week's material (`Week1/updateClonedRepo.ipynb`). New to everyone, so the whole class works through the same notebook.

In [ ]:
# Run this cell first. It lets the notebook find the course data folder,
# whether it runs from the week folder or the course folder.
import sys; sys.path.append(".."); from coursetools import data_file

### Concept 1: Saving a list with pickle

In Week 6 you saved **text**. To save a number, you turned it into text and back again. `pickle` saves almost any Python value exactly as it is, lists and dictionaries included.

Pickle files are not text, so the mode has a `b` for **binary**: `"wb"` to write, `"rb"` to read.

In [ ]:
import pickle

readings = [9.79, 9.83, 9.81, 9.80]

with open("readings.pkl", "wb") as file:
    pickle.dump(readings, file)

with open("readings.pkl", "rb") as file:
    loaded = pickle.load(file)

print(loaded)
print(type(loaded), type(loaded[0]))

It comes back as a list of floats, ready to use. No `float()` needed.

**Now break it.** Change `"wb"` to `"w"`. You get a `TypeError`: pickle writes bytes, not text.

### Concept 2: Saving objects

Objects pickle too. The one rule: **the class must be defined before you load**, because the file stores the data, not the design.

In [ ]:
import pickle


class Sample:
    def __init__(self, name, mass, volume):
        self.name = name
        self.mass = mass
        self.volume = volume

    def density(self):
        return self.mass / self.volume


samples = [Sample("Copper", 89.6, 10.0), Sample("Granite", 54.0, 20.0)]

with open("samples.pkl", "wb") as file:
    pickle.dump(samples, file)

with open("samples.pkl", "rb") as file:
    loaded = pickle.load(file)

for sample in loaded:
    print(f"{sample.name}: {sample.density():.2f} g/cm3")

The objects come back with their methods working.

**Now break it.** Restart the kernel (the circular arrow at the top), then run **only** the last three lines with the `pickle.load`. You need `import pickle` first, and then you get an `AttributeError`: Python cannot rebuild a `Sample` it has never seen.

### Concept 3: A cache that saves time

Reading 7,600 rows of CSV is quick here, but real datasets can take minutes. A **cache** saves the result the first time, then loads the saved copy on every later run. `try` and `except FileNotFoundError` from Week 5 decide which.

In [ ]:
import csv
import pickle
import matplotlib.pyplot as plt

try:
    with open("oxford_rain.pkl", "rb") as file:
        rain = pickle.load(file)
    print("Loaded from the cache")
except FileNotFoundError:
    rain = []
    with open(data_file("uk_weather.csv")) as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            if row[0] == "Oxford" and row[1] == "2025":
                rain.append(float(row[6]))
    with open("oxford_rain.pkl", "wb") as file:
        pickle.dump(rain, file)
    print("Read the CSV and saved the cache")

months = list(range(1, 13))
plt.bar(months, rain)
plt.xlabel("Month")
plt.ylabel("Rainfall (mm)")
plt.title("Oxford rainfall, 2025")
plt.show()

Run it twice. The first line printed changes, and the chart does not.

**Now break it.** Delete `oxford_rain.pkl` from the Week 8 folder and run again. The cache rebuilds itself. That is the point of the `except`.

### Now the challenges

Save this notebook with Ctrl and S, then open **`week8Challenges.ipynb`**.